# Zero-shot Baseline – Lyrics Generation

Generates **raw, zero-shot** outputs (no examples, no system prompt, no fine-tuning) for the candidate models:
Qwen2.5-3B, Llama-3.2-3B and Phi-3-mini, on `data/golden_dataset.json`.
Results are saved to `results/baseline_outputs.json`, to be evaluated later (count_syllables / extract_rhyme_tail are out of scope for this notebook).

**Recommended runtime:** GPU (a T4 is enough).

## 1. Install dependencies

In [1]:
%%capture
!pip install unsloth

## 2. Setup: Drive, paths, GPU

In [2]:
import json, gc, time, random
from pathlib import Path

import torch
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/LMusic")

DATA_PATH = PROJECT_ROOT / "data" / "golden_dataset.json"
RESULTS_PATH = PROJECT_ROOT / "results" / "baseline_outputs.json"
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)

assert DATA_PATH.exists(), f"Cannot find {DATA_PATH}. Check PROJECT_ROOT."
assert torch.cuda.is_available(), "No GPU active: Runtime > Change runtime type > GPU"
print("GPU:", torch.cuda.get_device_name(0))

Mounted at /content/drive
GPU: Tesla T4


## 3. Configuration

In [3]:
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()   # hides the repeated max_new_tokens / max_length warning

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)

# Unsloth mirrors of the official models (Llama needs no HF token / license acceptance this way)
MODELS = {
    "qwen2.5-3b":    "unsloth/Qwen2.5-3B-Instruct",
    "llama-3.2-3b":  "unsloth/Llama-3.2-3B-Instruct",
    "phi-3-mini":    "unsloth/Phi-3-mini-4k-instruct",
}

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True   # keep identical at fine-tuning time (QLoRA) so the baseline stays comparable

# Greedy decoding = reproducible. No system prompt: pure zero-shot, only the dataset prompt.
GEN_CONFIG = dict(max_new_tokens=300, do_sample=False)

## 4. Load golden dataset

In [4]:
with open(DATA_PATH, encoding="utf-8") as f:
    dataset = json.load(f)

print(f"{len(dataset)} prompts loaded")
print(dataset[0]["prompt"])

50 prompts loaded
4 versuri, rimă încrucișată, 8 silabe: un refren trist despre o dragoste care s-a terminat. Mă ajuți?


## 5. Generation functions

In [5]:
from unsloth import FastLanguageModel

def load_model(hf_name):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=hf_name,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,                 # auto (fp16 on T4, bf16 on newer GPUs)
        load_in_4bit=LOAD_IN_4BIT,
    )
    FastLanguageModel.for_inference(model)
    return model, tokenizer


def generate_one(model, tokenizer, prompt):
    messages = [{"role": "user", "content": prompt}]   # zero-shot: no system prompt
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, use_cache=True, **GEN_CONFIG)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def run_model(model_key, hf_name, dataset):
    print(f"\n=== {model_key} ({hf_name}) ===")
    model, tokenizer = load_model(hf_name)
    records = []
    t0 = time.time()
    for i, item in enumerate(dataset, 1):
        records.append({
            "id": item["id"],
            "model": model_key,
            "prompt": item["prompt"],
            "theme": item["theme"],
            "expected_constraints": item["expected_constraints"],
            "output": generate_one(model, tokenizer, item["prompt"]),
        })
        if i % 10 == 0:
            print(f"  {i}/{len(dataset)} ({time.time()-t0:.0f}s)")
    del model, tokenizer
    gc.collect(); torch.cuda.empty_cache()
    return records

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:2279: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1122 16:52:13.139000 421 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1122 16:52:13.178000 421 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 6. Run all models

Results are saved after each model; if the session is interrupted, models that already finished are not regenerated.

In [6]:
def save(all_records):
    payload = {
        "metadata": {
            "type": "baseline_zero_shot",
            "models": MODELS,
            "load_in_4bit": LOAD_IN_4BIT,
            "generation_config": GEN_CONFIG,
            "seed": SEED,
            "n_prompts": len(dataset),
        },
        "outputs": all_records,
    }
    with open(RESULTS_PATH, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

# resume from the existing file, if any
all_records = []
if RESULTS_PATH.exists():
    all_records = json.load(open(RESULTS_PATH, encoding="utf-8"))["outputs"]
done_models = {r["model"] for r in all_records}

for key, hf_name in MODELS.items():
    if key in done_models:
        print(f"Skipping {key} (already done)")
        continue
    all_records += run_model(key, hf_name, dataset)
    save(all_records)
    print(f"Saved to {RESULTS_PATH}")


=== qwen2.5-3b (unsloth/Qwen2.5-3B-Instruct) ===
==((====))==  Unsloth 2026.11.9: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

  10/50 (46s)
  20/50 (86s)
  30/50 (132s)
  40/50 (180s)
  50/50 (218s)
Saved to /content/drive/MyDrive/LMusic/results/baseline_outputs.json

=== llama-3.2-3b (unsloth/Llama-3.2-3B-Instruct) ===
==((====))==  Unsloth 2026.11.9: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

  10/50 (28s)
  20/50 (49s)
  30/50 (78s)
  40/50 (103s)
  50/50 (131s)
Saved to /content/drive/MyDrive/LMusic/results/baseline_outputs.json

=== phi-3-mini (unsloth/Phi-3-mini-4k-instruct) ===
==((====))==  Unsloth 2026.11.9: Fast Mistral patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

  10/50 (53s)
  20/50 (96s)
  30/50 (133s)
  40/50 (185s)
  50/50 (227s)
Saved to /content/drive/MyDrive/LMusic/results/baseline_outputs.json


## 7. Quick check

In [7]:
from collections import Counter
print(Counter(r["model"] for r in all_records))

for r in all_records:
    if r["id"] in (1, 2):
        print(f"\n--- [{r['model']}] id={r['id']} ---")
        print(r["output"])

Counter({'qwen2.5-3b': 50, 'llama-3.2-3b': 50, 'phi-3-mini': 50})

--- [qwen2.5-3b] id=1 ---
Pentru a crea o poezie cu criteriile cerute (rimă încrucișată, 4 versuri, 8 silabe, și un refren), voi încerca să generez o poezie care să reflecteze o dragoste care s-a terminat într-un ton trist. Iată ce am obținut:

Refren:
Trăsăturile tău rămân,
În cor la mine se întâlnesc.
Dragoste ce s-a sfârșit,
Sunt mereu în mine.

1. Versul 1:
Mălțuiți de nevoi tăi,
Nemultumit, împărtășit.
Câteva zile, păcii rămâne,
Dar nici mărețe nesfârșite.

2. Versul 2:
Furtunul tău, ca un vis,
În cor meu, se încriniște.
Nimic nu mă încurajează,
Pe viața noastră, să rămânem.

3. Versul 3:
Lumea neașteptată, pe lângă noi,
A scos-n, într-o altă lume.
Nemultumit, împărtăș

--- [qwen2.5-3b] id=2 ---
Bună! Iată o piesă pop pentru vară în limba română, folosind criteriile tale, cu un ritm alert și 10 silabe pe vers:

Vacanța se sfârște, zilele scurte,
Într-o lume de frumusețe, împărtășită.
Pentru cea mai bună vacanță, am